# 5FTC2068 ARAIP – Project Part 2: CIFAR-10 CNN for the Webots e-puck
**Author:** Aisur Omar  
Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

Sections: 1 Setup · 2 Data · 3 Model (all layers) · 4 Training · 5 Accuracy & error analysis · 6 Three own images · 7 Webots camera image · 8 Optimisation (TFLite quantisation, latency) · 9 Export

In [ ]:
import numpy as np, matplotlib.pyplot as plt, tensorflow as tf, time, os
from tensorflow import keras
from tensorflow.keras import layers
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))
tf.random.set_seed(2068); np.random.seed(2068)
CLASSES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]

## 2. Load and prepare CIFAR-10 (50,000 train / 10,000 test, 32×32×3)

In [ ]:
# CIFAR-10 from the Hugging Face mirror (much faster than the Toronto server)
from datasets import load_dataset
ds = load_dataset("uoft-cs/cifar10")
x_train = np.stack([np.array(i) for i in ds["train"]["img"]]); y_train = np.array(ds["train"]["label"])
x_test = np.stack([np.array(i) for i in ds["test"]["img"]]); y_test = np.array(ds["test"]["label"])
y_train, y_test = y_train.flatten(), y_test.flatten()
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0
print(x_train.shape, x_test.shape)
plt.figure(figsize=(10,2))
for i in range(10):
    plt.subplot(1,10,i+1); plt.imshow(x_train[np.where(y_train==i)[0][0]]); plt.title(CLASSES[i],fontsize=8); plt.axis("off")
plt.show()

## 3. CNN architecture – all layers
Three convolution blocks (32→64→128 filters), each Conv–BN–ReLU ×2 + MaxPool + Dropout, then a small dense head. Data augmentation layers are part of the model so they only run during training.

In [ ]:
augment = keras.Sequential([layers.RandomFlip("horizontal"),
                            layers.RandomTranslation(0.1, 0.1),
                            layers.RandomRotation(0.05)], name="augmentation")
def conv_block(x, f, drop):
    for _ in range(2):
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    return layers.Dropout(drop)(x)

inp = keras.Input(shape=(32,32,3), name="image")
x = augment(inp)
x = conv_block(x, 32, 0.2)
x = conv_block(x, 64, 0.3)
x = conv_block(x, 128, 0.4)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.5)(x)
out = layers.Dense(10, activation="softmax", name="class_probs")(x)
model = keras.Model(inp, out, name="cifar10_cnn")
model.summary()
keras.utils.plot_model(model, "cnn_layers.png", show_shapes=True, dpi=70)

## 4. Training (Adam, cosine learning-rate decay, early stopping)

In [ ]:
EPOCHS, BATCH = 40, 128
steps = (len(x_train)//BATCH) * EPOCHS
model.compile(optimizer=keras.optimizers.Adam(keras.optimizers.schedules.CosineDecay(1e-3, steps)),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
t0 = time.time()
hist = model.fit(x_train, y_train, validation_split=0.1, epochs=EPOCHS, batch_size=BATCH,
                 callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)], verbose=2)
print("Training time: %.1f min" % ((time.time()-t0)/60))
fig, ax = plt.subplots(1,2, figsize=(11,3.5))
ax[0].plot(hist.history["accuracy"], label="train"); ax[0].plot(hist.history["val_accuracy"], label="val"); ax[0].set_title("Accuracy"); ax[0].legend()
ax[1].plot(hist.history["loss"], label="train"); ax[1].plot(hist.history["val_loss"], label="val"); ax[1].set_title("Loss"); ax[1].legend()
plt.savefig("training_curves.png", dpi=120, bbox_inches="tight"); plt.show()

## 5. Accuracy score and error analysis

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"TEST ACCURACY: {test_acc:.4f}   (test loss {test_loss:.4f})")
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
y_pred = model.predict(x_test, verbose=0).argmax(1)
print(classification_report(y_test, y_pred, target_names=CLASSES, digits=3))
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, xticks_rotation=45, colorbar=False, cmap="Blues")
plt.title(f"Confusion matrix (test acc {test_acc:.3f})"); plt.savefig("confusion_matrix.png", dpi=120, bbox_inches="tight"); plt.show()
off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print(f"Most common confusion: {CLASSES[i]} predicted as {CLASSES[j]} ({off[i,j]} images)")

## 6. Prediction on three of my own images
Run the cell and upload 3 photos (e.g. a car, a cat, an aeroplane).

In [ ]:
from google.colab import files
from PIL import Image
def predict_file(path, mdl=None):
    mdl = mdl or model
    img = Image.open(path).convert("RGB")
    w, h = img.size; s = min(w, h)
    img_sq = img.crop(((w-s)//2, (h-s)//2, (w-s)//2+s, (h-s)//2+s)).resize((32,32), Image.BILINEAR)
    x = np.asarray(img_sq, dtype="float32")[None]/255.0
    p = mdl.predict(x, verbose=0)[0]
    return img, p
def show_predictions(paths, title):
    plt.figure(figsize=(4.2*len(paths), 4.4))
    for k, pth in enumerate(paths):
        img, p = predict_file(pth)
        top = p.argsort()[::-1][:3]
        plt.subplot(1, len(paths), k+1); plt.imshow(img); plt.axis("off")
        plt.title(f"{CLASSES[top[0]]} {p[top[0]]*100:.1f}%\n" + ", ".join(f"{CLASSES[t]} {p[t]*100:.0f}%" for t in top[1:]), fontsize=10)
    plt.suptitle(title); plt.savefig(title.replace(" ","_")+".png", dpi=120, bbox_inches="tight"); plt.show()
own = files.upload()
show_predictions(list(own.keys())[:3], "Own image predictions")

## 7. Prediction on the Webots e-puck camera image
Upload the PNGs saved by the controller in `captures/` (e.g. `front_object_*.png` = cat, `rear_object_*.png` = dog).

In [ ]:
webots = files.upload()
show_predictions(list(webots.keys()), "Webots camera predictions")

## 8. Optimisation for on-robot use
Convert to **TensorFlow Lite** with (a) float16 and (b) full-integer int8 post-training quantisation, then compare size, accuracy and per-image latency (single CPU thread, like a robot's embedded processor).

In [ ]:
model.save("cifar10_cnn.keras")
conv = tf.lite.TFLiteConverter.from_keras_model(model)
tfl_fp32 = conv.convert()
conv = tf.lite.TFLiteConverter.from_keras_model(model); conv.optimizations=[tf.lite.Optimize.DEFAULT]; conv.target_spec.supported_types=[tf.float16]
tfl_fp16 = conv.convert()
def rep():
    for i in range(300): yield [x_train[i:i+1]]
conv = tf.lite.TFLiteConverter.from_keras_model(model); conv.optimizations=[tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep
conv.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.uint8; conv.inference_output_type = tf.uint8
tfl_int8 = conv.convert()
for n, b in [("cifar10_cnn.tflite", tfl_fp32), ("cifar10_cnn_fp16.tflite", tfl_fp16), ("cifar10_cnn_int8.tflite", tfl_int8)]:
    open(n, "wb").write(b)

def eval_tflite(blob, n=2000):
    it = tf.lite.Interpreter(model_content=blob, num_threads=1); it.allocate_tensors()
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    correct, times = 0, []
    for k in range(n):
        x = x_test[k:k+1]
        if i["dtype"] == np.uint8:
            s, z = i["quantization"]; x = (x/s + z).astype(np.uint8)
        t = time.perf_counter(); it.set_tensor(i["index"], x); it.invoke(); times.append(time.perf_counter()-t)
        correct += it.get_tensor(o["index"])[0].argmax() == y_test[k]
    return correct/n, 1000*np.median(times)

keras_size = os.path.getsize("cifar10_cnn.keras")/1024
t = time.perf_counter()
for k in range(200): model(x_test[k:k+1], training=False)
keras_ms = (time.perf_counter()-t)/200*1000
rows = [("Keras FP32", keras_size, model.evaluate(x_test[:2000], y_test[:2000], verbose=0)[1], keras_ms)]
for name, blob in [("TFLite FP32", tfl_fp32), ("TFLite FP16", tfl_fp16), ("TFLite INT8", tfl_int8)]:
    acc, ms = eval_tflite(blob); rows.append((name, len(blob)/1024, acc, ms))
print(f"{'Model':<12}{'Size KB':>9}{'Acc':>8}{'ms/img':>8}{'max FPS':>9}")
for r in rows: print(f"{r[0]:<12}{r[1]:>9.0f}{r[2]:>8.3f}{r[3]:>8.2f}{1000/r[3]:>9.0f}")
fig, ax = plt.subplots(1,2, figsize=(10,3.2))
ax[0].bar([r[0] for r in rows], [r[1] for r in rows], color="#4a78c2"); ax[0].set_title("Model size (KB)")
ax[1].bar([r[0] for r in rows], [r[3] for r in rows], color="#e08a2c"); ax[1].set_title("Latency per image (ms, 1 CPU thread)")
for a in ax: a.tick_params(axis="x", rotation=20)
plt.savefig("optimisation_comparison.png", dpi=120, bbox_inches="tight"); plt.show()

## 9. Export model + figures (copy the `.tflite` into the Webots project's `models/` folder)

In [ ]:
import zipfile
with zipfile.ZipFile("cnn_outputs.zip", "w") as z:
    for f in os.listdir("."):
        if f.endswith((".png", ".tflite", ".keras")): z.write(f)
files.download("cnn_outputs.zip")

## 10. Webots images with an object crop (cheap region proposal) + summary of all predictions

In [ ]:
from PIL import Image as PILImage
def crop_object(path):
    a = np.asarray(PILImage.open(path).convert('RGB')).astype(int)
    m = (a.max(2) - a.min(2)) > 40          # saturated pixels = the object (sky/floor are grey)
    ys, xs = np.where(m)
    cy, cx = (ys.min() + ys.max()) // 2, (xs.min() + xs.max()) // 2
    s = int(max(ys.max() - ys.min(), xs.max() - xs.min()) * 1.15) // 2 + 1
    out = 'crop_' + path
    PILImage.fromarray(a.astype('uint8')).crop((cx - s, cy - s, cx + s, cy + s)).save(out)
    return out
show_predictions([crop_object(n) for n in webots.keys()], 'Webots camera predictions with object crop')
for n in list(own.keys())[:3] + list(webots.keys()) + ['crop_' + k for k in webots.keys()]:
    p = predict_file(n)[1]; print(n, '->', CLASSES[p.argmax()], round(100 * float(p.max()), 1), '%')
for r in rows: print(r[0], round(r[1]), 'KB', round(r[2]*100, 2), '%', round(r[3], 2), 'ms', round(1000/r[3]), 'FPS')
z = zipfile.ZipFile('cnn_outputs.zip', 'w'); [z.write(f) for f in os.listdir('.') if f.endswith(('.png', '.tflite', '.keras'))]; z.close()
files.download('cnn_outputs.zip')